# The Route to Profit
## An Aviation Investment Analysis — Zoltan Wilhelm

**Central question:** *Which airline and region is the most investment-worthy?*

Five pieces of evidence:
1. **WHERE** — Which region leads on size, margin and growth?
2. **WHAT** — Legacy or low-cost: which model performs better?
3. **WHEN** — How fast does the market recover from a shock?
4. **WHICH** — Which routes carry the revenue?
5. **HOW** — Volume or margin: where is the real return?

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# larger, readable chart defaults
plt.rcParams.update({
    'font.size': 13, 'axes.titlesize': 16, 'axes.titleweight': 'bold',
    'axes.labelsize': 13, 'legend.fontsize': 11, 'figure.figsize': (11, 6),
})

NAVY, AMBER, MUTE1, MUTE2 = '#1E2761', '#D98E04', '#3B5BA5', '#8FA3D4'

### Load the data
Upload `airline_financials.csv` and `route_performance.csv` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()

af = pd.read_csv('airline_financials.csv')   # airline-level financials
rp = pd.read_csv('route_performance.csv')    # route-level performance

print('Financials:', af.shape, '| Routes:', rp.shape)

## 2. Data Validation & Cleaning

Two deliberate exclusions, both defensible:
- **2026** — an incomplete year that would distort every trend
- **Latin America** — only 2 airlines (LATAM, Azul), too small a sample for a fair regional average

In [ ]:
# --- quality checks before touching anything ---
print('Missing values (financials):', af.isnull().sum().sum())
print('Missing values (routes):    ', rp.isnull().sum().sum())
print('Duplicate rows:', af.duplicated().sum(), '/', rp.duplicated().sum())
print('\nYears present:', sorted(af['year'].unique()))
print('Regions:', sorted(af['region'].unique()))
print('Business models:', sorted(af['business_model'].unique()))

In [ ]:
# --- apply the two exclusions ---
af = af[(af['year'] != 2026) & (af['region'] != 'Latin America')].copy()
rp = rp[rp['year'] != 2026].copy()

# derived metric: revenue efficiency per kilometre flown
rp['fare_per_km'] = rp['avg_fare_usd'] / rp['distance_km']

years = sorted(af['year'].unique())
print('After cleaning:', af.shape, '| years', years[0], '-', years[-1])
print('Airlines:', af['airline_name'].nunique(), '| Regions:', af['region'].nunique())

### Business model classification

`Alaska Airlines` is labelled *regional* in the source data — a category with only one member.
By revenue scale (~$11B vs Delta's ~$65B) and margin profile (~8% vs legacy's ~6%) it behaves
like a low-cost carrier, so it is grouped there.

In [ ]:
LEGACY  = ['Delta Air Lines', 'United Airlines', 'American Airlines']
LOWCOST = ['Southwest Airlines', 'JetBlue', 'Alaska Airlines']

na = af[af['region'] == 'North America'].copy()
na['model'] = np.where(na['airline_name'].isin(LEGACY), 'Legacy', 'Low-cost')

na.groupby('model')['airline_name'].unique()

## 3. WHERE — Which region leads on size, margin and growth?

Size alone is not an investment case. A region could be large simply because it is mature.
So this question is tested three ways: **revenue**, **operating margin**, and **growth rate**.

In [ ]:
# revenue trajectory per region
rev = af.groupby(['region','year'])['revenue_usd_bn'].mean().unstack(0)

fig, ax = plt.subplots()
for region in ['North America','Europe','Asia','Middle East']:
    is_hero = region == 'North America'
    ax.plot(rev.index, rev[region],
            color=AMBER if is_hero else (MUTE1 if region=='Europe' else MUTE2),
            linewidth=3 if is_hero else 2,
            label=region, zorder=3 if is_hero else 2)

ax.set_title('Average Airline Revenue by Region')
ax.set_xlabel('Year'); ax.set_ylabel('Avg. Revenue (USD Bn)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- the three proofs, side by side ---
# margin measured on normal years only, so COVID does not distort the comparison
normal = af[~af['year'].isin([2020, 2021])]
margin = normal.groupby('region')['operating_margin_pct'].mean()

# CAGR = compound annual growth rate from first year to last
n_years = years[-1] - years[0]
cagr = ((rev.iloc[-1] / rev.iloc[0]) ** (1/n_years) - 1) * 100

summary = pd.DataFrame({
    'revenue_2025_bn': rev.iloc[-1].round(2),
    'avg_margin_pct':  margin.round(2),
    'cagr_pct':        cagr.round(2),
}).sort_values('revenue_2025_bn', ascending=False)

summary

**Takeaway:** North America leads on *all three* measures — largest revenue, highest margin (8.1%
vs Europe's 7.3%), and fastest growth (5.9%/yr). Three independent proofs pointing the same way.
The rest of the analysis focuses here.

## 4. WHAT — Legacy or low-cost: which model performs better?

Comparing the two categories head to head, rather than airline by airline.

In [ ]:
model_rev = na.groupby(['model','year'])['revenue_usd_bn'].mean().unstack(0)

fig, ax = plt.subplots()
ax.plot(model_rev.index, model_rev['Legacy'],   color=NAVY,  linewidth=3, marker='o', label='Legacy (avg)')
ax.plot(model_rev.index, model_rev['Low-cost'], color=AMBER, linewidth=3, marker='o', label='Low-cost (avg)')
ax.set_title('Average Revenue: Legacy vs Low-Cost')
ax.set_xlabel('Year'); ax.set_ylabel('Avg. Revenue (USD Bn)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# the twist: revenue lead vs margin lead point in OPPOSITE directions
na_normal = na[~na['year'].isin([2020, 2021])]
print('Revenue 2025  — Legacy: $%.1fB  |  Low-cost: $%.1fB  (%.1fx lead)' % (
    model_rev['Legacy'].iloc[-1], model_rev['Low-cost'].iloc[-1],
    model_rev['Legacy'].iloc[-1] / model_rev['Low-cost'].iloc[-1]))
print('Avg margin    — Legacy: %.1f%%   |  Low-cost: %.1f%%' % (
    na_normal[na_normal['model']=='Legacy']['operating_margin_pct'].mean(),
    na_normal[na_normal['model']=='Low-cost']['operating_margin_pct'].mean()))

**Takeaway:** legacy carriers earn **3.5x more revenue**, but low-cost carriers are **nearly twice
as profitable per dollar** (10.4% vs 5.7% margin). Scale and efficiency point in different directions —
a tension the final recommendation has to resolve.

## 5. WHEN — How fast does the market recover from a shock?

If recovery is fast, a crash is a buying window. If it is slow, it is a permanent impairment.
This is measured as year-over-year change in North American revenue.

In [ ]:
na_yr = na.groupby('year')['revenue_usd_bn'].mean()

timetable = pd.DataFrame({'avg_revenue_bn': na_yr.round(2)})
timetable['yoy_change_pct'] = (na_yr.pct_change() * 100).round(1)
timetable['phase'] = ''
labels = {2019:'Pre-COVID peak', 2020:'Collapse', 2021:'Recovery begins',
          2022:'Breakout year', 2023:'Normalising', 2024:'Steady', 2025:'Steady'}
for y, lbl in labels.items():
    if y in timetable.index:
        timetable.loc[y, 'phase'] = lbl

timetable.loc[2019:]

In [ ]:
# visualise the swing
chg = na_yr.pct_change() * 100
sub = chg.loc[2019:]

fig, ax = plt.subplots(figsize=(10,5.5))
colors = ['#B23A3A' if v < 0 else (AMBER if v > 50 else MUTE1) for v in sub]
bars = ax.bar(sub.index.astype(str), sub.values, color=colors)
for b, v in zip(bars, sub.values):
    ax.annotate(f'{v:+.1f}%', xy=(b.get_x()+b.get_width()/2, v),
                xytext=(0, 6 if v > 0 else -18), textcoords='offset points',
                ha='center', fontsize=11, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('North America: Year-over-Year Revenue Change')
ax.set_ylabel('Change (%)'); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

**Takeaway:** the market fell **62%** in 2020, rebounded **+64.9%** in 2021 and **+73.7%** in 2022 —
the single largest annual gain in the dataset, and the year it passed its pre-COVID peak.
Full recovery took roughly **two years**, then growth settled to a healthy 3-6%/yr.
That bounded downside is what makes a crash a buying window rather than a permanent loss.

## 6. WHICH — Which routes carry the revenue?

In [ ]:
NA_REGIONS = ['North America', 'Atlantic', 'Pacific']
r25 = rp[(rp['region'].isin(NA_REGIONS)) & (rp['year'] == 2025)].copy()
r25['type'] = np.where(r25['region'] == 'North America', 'Domestic', 'International')
r25 = r25.sort_values('annual_revenue_usd_m')

fig, ax = plt.subplots(figsize=(11, 7))
colors = [MUTE1 if t == 'Domestic' else AMBER for t in r25['type']]
bars = ax.barh(r25['route'], r25['annual_revenue_usd_m'], color=colors)
for b, v in zip(bars, r25['annual_revenue_usd_m']):
    ax.annotate(f'${v:,.0f}M', xy=(v, b.get_y()+b.get_height()/2),
                xytext=(5, 0), textcoords='offset points', va='center', fontsize=11)

ax.set_title('North America-Related Route Revenue, 2025')
ax.set_xlabel('Annual Revenue (USD Millions)')
ax.set_xlim(0, r25['annual_revenue_usd_m'].max()*1.18)
ax.grid(alpha=0.3, axis='x')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=AMBER, label='International'),
                   Patch(color=MUTE1, label='Domestic')], loc='lower right')
plt.tight_layout(); plt.show()

**Takeaway:** the three highest-earning routes are all **trans-Pacific international**, each above
$1 billion a year. Domestic routes are solid but sit mid-pack. If total revenue were the only
measure, international would win outright — but the next question complicates that.

## 7. HOW — Which route type earns more per km?

Total revenue rewards long routes automatically, since a longer flight sells a more expensive ticket.
Normalising to **revenue per kilometre** removes that distance advantage and exposes which segment
is genuinely more efficient.

*Caveat, stated openly: the dataset holds no route-level cost data, so fare-per-km is an
efficiency **proxy**, not a true profit figure.*

In [ ]:
carriers = {'Delta Air Lines':'Delta', 'United Airlines':'United',
            'American Airlines':'American', 'JetBlue':'JetBlue'}

rows = []
for full, short in carriers.items():
    ar = rp[rp['main_airlines'].str.contains(short, na=False)]
    dom  = ar[ar['region'] == 'North America']['fare_per_km'].mean()
    intl = ar[ar['region'].isin(['Atlantic','Pacific'])]['fare_per_km'].mean()
    rows.append({'airline': short,
                 'domestic_usd_km': round(dom, 4) if pd.notna(dom) else np.nan,
                 'intl_usd_km':     round(intl, 4) if pd.notna(intl) else np.nan,
                 'domestic_edge_pct': round((dom/intl - 1)*100, 1) if pd.notna(intl) else np.nan})

efficiency = pd.DataFrame(rows)
efficiency

In [ ]:
plot_df = efficiency.dropna(subset=['intl_usd_km'])
x = np.arange(len(plot_df)); w = 0.36

fig, ax = plt.subplots(figsize=(9.5, 6))
b1 = ax.bar(x - w/2, plot_df['domestic_usd_km'], w, color=MUTE1, label='Domestic')
b2 = ax.bar(x + w/2, plot_df['intl_usd_km'],     w, color=AMBER, label='International')

for bars, color in [(b1, MUTE1), (b2, '#9B6500')]:
    for b in bars:
        ax.annotate(f'${b.get_height():.3f}', xy=(b.get_x()+b.get_width()/2, b.get_height()),
                    xytext=(0, 5), textcoords='offset points', ha='center',
                    fontsize=11, fontweight='bold', color=color)

# edge % label above each pair — the headline number for this slide
for i, row in plot_df.reset_index(drop=True).iterrows():
    ax.text(i, max(row['domestic_usd_km'], row['intl_usd_km']) + 0.0035,
            f'+{row["domestic_edge_pct"]}%', ha='center', fontsize=10,
            color='#444444', fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(plot_df['airline'])
ax.set_title('Which Route Type Earns More Per km?')
ax.set_ylabel('Revenue per km (USD)')
ax.set_ylim(0, 0.075)
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

**Takeaway:** domestic routes earn **33-45% more per kilometre** than international ones, for every
airline measured. Combined with the previous question: international wins on **volume**, domestic
wins on **efficiency**. That trade-off is the central finding of the analysis.

## 8. Added bonus — revenue growth rates, 2015–2025

Not one of the five core questions, but a pattern worth flagging: indexing every carrier's revenue
to its 2015 level (2015 = 100) shows the shape of each airline's growth, not just its size.

In [ ]:
base_year = 2015
rev_by_airline_yr = na.pivot_table(index='year', columns='airline_name', values='revenue_usd_bn')

indexed = rev_by_airline_yr / rev_by_airline_yr.loc[base_year] * 100

line_colors = {'Alaska Airlines':'#1E2761', 'Delta Air Lines':'#3B7DD8', 'JetBlue':'#5BA3C4',
               'United Airlines':'#7A9AD0', 'Southwest Airlines':'#A88FD0', 'American Airlines':'#9AB5E0'}

fig, ax = plt.subplots(figsize=(11, 6))
order = indexed.iloc[-1].sort_values(ascending=False).index
for airline in order:
    ax.plot(indexed.index, indexed[airline], color=line_colors[airline],
            linewidth=2.4, alpha=0.95, label=airline)

ax.axhline(100, color='#CCCCCC', linewidth=1, linestyle='--')
ax.text(base_year + 0.1, 104, f'{base_year} baseline', fontsize=9.5, color='#AAAAAA')
ax.axvspan(2020, 2021, alpha=0.07, color='#AA3333')
ax.text(2020.5, 15, 'COVID', ha='center', fontsize=9.5, color='#AA4444', style='italic')

ax.set_title(f'Indexed Revenue Growth, {base_year}–2025 ({base_year} = 100)')
ax.set_xlabel('Year'); ax.set_ylabel(f'Revenue Index ({base_year} = 100)')
ax.set_ylim(0, 240)
ax.legend(loc='lower right', ncol=2, framealpha=0.95)
ax.grid(alpha=0.18)
plt.tight_layout(); plt.show()

**Takeaway:** Alaska grew fastest at **7.6% CAGR** — doubling revenue since 2015 while the larger
carriers grew 30–55% over the same span. Smallest airline, highest growth rate — not one of the
five core questions, but too notable a pattern to leave out.

## 9. The Recommendation

Ranking the six North American carriers across the measures that matter to an investor.

In [ ]:
ranking = na_normal.groupby('airline_name').agg(
    model=('model','first'),
    avg_margin_pct=('operating_margin_pct','mean'),
).round(1)

rev_by_airline = na.pivot_table(index='airline_name', columns='year', values='revenue_usd_bn')
ranking['revenue_2025_bn'] = rev_by_airline[2025].round(1)
ranking['cagr_pct'] = (((rev_by_airline[years[-1]] / rev_by_airline[years[0]]) ** (1/n_years) - 1) * 100).round(1)
ranking['recovery_2021_pct'] = (rev_by_airline[2021] / rev_by_airline[2019] * 100).round(0)

ranking.sort_values('revenue_2025_bn', ascending=False)

### Invest in North America.

| Role | Airline | Metric |
|---|---|---|
| **The core** | Delta Air Lines | $64.7B revenue — largest and steadiest |
| **The margin play** | Southwest Airlines | 11.7% operating margin — best in class |

**The call:** hold Delta for scale, Southwest for margin — and watch Alaska quietly outgrow them all.

---

### The surprising find — Alaska Airlines

Not one of the two core picks, but a real pattern in the data: Alaska is the **smallest** of the six
carriers by revenue, yet posts the **fastest growth (7.6% CAGR)** and an **8.3% average margin** —
above all three legacy carriers. Smaller base, real momentum.

---

### Limitations
- No route-level cost data — profitability is measured by proxy
- Airline and route data are joined by year, a deliberate simplification
- Route-level data covers only 4 of the 6 North American carriers
- Single-source dataset, so figures are not cross-validated